# 02 — Strategy backtest (Phase 4)

**Design period only (2024-08-01 to 2026-06-30).** Every number comes from `src/backtest/`
(`engine.py`, `costs.py`, `metrics.py`); this notebook only calls it and displays the results.

Rules (DECISIONS.md 2026-11-04 and 2026-11-08, all values in `config.yaml`):
signal = trailing 7-day mean of the known-at-t gap; enter when |signal| > 10% APR, short the
higher-funding venue and long the other; exit when |signal| < 3% APR or on a sign change;
spread stop at 100 bps against the entry spread, then no re-entry until the signal meets the exit
rule; $10,000 per leg, 3x leverage, one position per coin, up to 15; rebalance when a leg's margin
falls below 50% of initial ($1 fee, 2h delay); liquidation at per-venue maintenance margins with a
100 bps penalty. Decisions at bar closes (every 4h before each coin's hourly Hyperliquid history
starts on 2026-04-09, hourly after), fills at the next bar's open, taker on both legs.
Returns are on total capital: $100,000 of margin plus a 50% buffer = $150,000.

The random-entry benchmark (200 seeds) is too slow to rerun here; its summary is read from
`report/tables/backtest_summary.csv`, written by `python -m src.backtest.engine`.

In [1]:
import pandas as pd
from IPython.display import display
from src.config import load_config, ROOT
from src.backtest import engine as E, metrics as M
from src.analysis import backtest_figures as F

cfg = load_config()
inputs = E.load_inputs(cfg)
base = E.run(cfg, inputs)
zc = E.run(E.zero_cost(cfg), inputs)
always = E.run(cfg, inputs, E.AlwaysOnPolicy(1))
usd = "{:,.0f}".format

## 1. Headline metrics

Strategy against the three benchmarks of `docs/plan.md` Phase 4 item 7: zero cost (same rules, no
fees or slippage), always-on (short Hyperliquid / long Binance in every coin for the whole period,
exits only on liquidation) and random entry (same number of trades, holding periods and sides per
coin at random times; mean and 5-95% range over 200 seeds).

In [2]:
summary = pd.read_csv(ROOT / "report/tables/backtest_summary.csv", index_col=0)
fmt = {r: "{:.2%}" for r in ["ann_return_total_capital", "ann_return_deployed", "max_drawdown_frac", "hit_rate"]}
fmt.update({r: "{:,.0f}" for r in ["total_pnl", "capital", "avg_deployed_capital", "max_drawdown_usd", "worst_trade"]})
fmt.update({r: "{:.2f}" for r in ["sharpe", "avg_hold_days"]})
fmt.update({r: "{:.0f}" for r in ["trades", "spread_stops", "liquidations", "rebalances"]})
summary.apply(lambda row: row.map(fmt.get(row.name, "{:.2f}").format), axis=1)

,strategy,zero_cost,always_on,random_mean,random_p05,random_p95
total_pnl,"-8,871",99,"-10,928","-13,394","-18,734","-7,485"
capital,"150,000","150,000","150,000","150,000","150,000","150,000"
ann_return_total_capital,-3.09%,0.03%,-3.80%,-4.66%,-6.52%,-2.61%
avg_deployed_capital,"32,170","31,645","99,943","28,929","27,778","30,027"
ann_return_deployed,-14.40%,0.16%,-5.71%,-24.21%,-34.84%,-13.71%
sharpe,-2.02,0.02,-0.66,-2.29,-3.40,-1.16
max_drawdown_usd,"-8,878","-2,230","-12,527","-14,009","-18,838","-9,468"
max_drawdown_frac,-5.92%,-1.48%,-8.36%,-9.31%,-12.54%,-6.18%
trades,213,213,57,214,214,214
hit_rate,31.92%,76.53%,35.09%,17.95%,14.49%,21.03%


## 2. P&L decomposition (the main exhibit)

Each trade's P&L is booked in separate components that sum exactly to its total: Hyperliquid
funding (every hour held), Binance funding (only at settlements while open), price spread (both
legs at unslipped fill prices), fees, slippage, collateral-transfer fees and liquidation penalties.
`price_unrealised` is the one position still open at the end of the design period.

In [3]:
dec = pd.DataFrame({"strategy": M.decomposition(base.trades).loc["ALL"],
                    "zero_cost": M.decomposition(zc.trades).loc["ALL"],
                    "always_on": M.decomposition(always.trades).loc["ALL"]})
display(dec.style.format(usd))
F.plot_decomposition(dec["strategy"], "Strategy P&L by component, design period ($)",
                     ROOT / "report/figures/backtest_decomposition.png");

,strategy,zero_cost,always_on
hl_funding,"12,325","11,760","23,149"
bn_funding,"-2,075","-1,891","-6,402"
price_spread,"-3,706","-4,212","-18,255"
fees,"-3,815",0,"-1,013"
slippage,"-6,042",0,"-1,607"
transfer_fees,-196,-188,-329
liquidation,"-5,361","-5,369","-6,540"
price_unrealised,-1,-1,69
total,"-8,871",99,"-10,928"


### Per coin

In [4]:
M.decomposition(base.trades).style.format(usd)

,hl_funding,bn_funding,price_spread,fees,slippage,transfer_fees,liquidation,price_unrealised,total
coin,,,,,,,,,
TIA,196,323,"-1,667",-297,-498,-13,-437,0,"-2,392"
SUI,926,-276,-954,-289,-485,-19,-607,0,"-1,703"
WLD,384,135,-296,-352,-592,-15,-524,0,"-1,260"
kPEPE,979,-28,525,-299,-502,-20,-469,-1,186
AVAX,"1,150",-287,-397,-309,-521,-11,-235,0,-611
TAO,"1,148",-68,"-1,671",-288,-483,-21,-619,0,"-2,002"
kBONK,306,5,-159,-301,-506,-9,-470,0,"-1,133"
APT,485,-21,769,-346,-582,-15,-354,0,-65
SOL,"1,629",-570,-536,-228,-382,-18,-192,0,-297


## 3. Equity curves

In [5]:
F.plot_equity({"strategy": base.daily, "zero cost": zc.daily, "always-on": always.daily},
              base.capital, ROOT / "report/figures/backtest_equity.png");

## 4. How trades end

Exit reasons per coin, with liquidations and rebalances. A liquidation closes one leg at its
liquidation price less the 100 bps penalty; the other leg closes as a taker at the next bar open.

In [6]:
t = base.trades
reasons = pd.crosstab(t["coin"], t["exit_reason"]).reindex(cfg["universe"]["coins"]).fillna(0).astype(int)
reasons["rebalances"] = t.groupby("coin")["n_rebalances"].sum().reindex(reasons.index).fillna(0).astype(int)
reasons["avg_hold_days"] = ((t["exit_hour"] - t["entry_hour"]) / 86_400_000).groupby(t["coin"]).mean().reindex(reasons.index).round(1)
reasons

exit_reason,below_exit,liquidation,open_at_end,sign_change,rebalances,avg_hold_days
coin,,,,,,
BTC,7,0,0,0,1,29.8
ETH,8,0,0,0,4,18.0
SOL,10,1,0,1,18,21.3
kPEPE,13,4,1,0,20,16.1
SUI,12,4,0,0,19,12.1
WIF,10,7,0,0,12,11.8
TIA,13,4,0,0,13,13.2
DOGE,7,2,0,0,13,17.5
WLD,15,5,0,0,15,10.1


### Liquidated trades by component

What a liquidated trade costs compared with trades that end on the signal.

In [7]:
grp = t.assign(ended=t["exit_reason"].where(t["exit_reason"] == "liquidation", "other"))
grp.groupby("ended")[E.COMPONENTS + ["total"]].agg(["sum", "mean"]).T.unstack().style.format(usd)

## 5. Sanity checks

Components sum to each trade's total and to the final equity; fills happen only at bar starts;
Hyperliquid funding is booked for exactly the hours held (liquidated trades excluded); counts of
missing funding hours (booked as zero) and deferred fills.

In [8]:
pd.DataFrame({"strategy": M.sanity(base, inputs), "zero_cost": M.sanity(zc, inputs),
              "always_on": M.sanity(always, inputs)})

,strategy,zero_cost,always_on
missing_hl_funding,7.000000e+00,7.000000e+00,3.000000e+01
missing_bn_funding,0.000000e+00,0.000000e+00,0.000000e+00
deferred_fills,0.000000e+00,0.000000e+00,0.000000e+00
liquidations,4.500000e+01,4.500000e+01,5.700000e+01
rebalances,1.960000e+02,1.880000e+02,3.290000e+02
rebalance_no_surplus,0.000000e+00,0.000000e+00,0.000000e+00
spread_stops,0.000000e+00,0.000000e+00,0.000000e+00
max_abs_component_sum_error,0.000000e+00,0.000000e+00,0.000000e+00
equity_vs_trades_error,1.127773e-10,1.000444e-11,6.493792e-10
fills_off_bar_start,0.000000e+00,0.000000e+00,0.000000e+00
